In [1]:
import os

from transformers import DistilBertTokenizerFast, TrainingArguments, Trainer

from llm.data_utils import read_dataset, split_dataframe
from utils import get_project_root

import numpy as np
import evaluate

D:\projects\mldl\semevalpolar


In [2]:
tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')

In [3]:
df_data = read_dataset(os.path.join(get_project_root(), 'data', 'dev_phase', 'subtask1', 'train', 'eng.csv'))

In [4]:
train_df, val_df, test_df = split_dataframe(df_data)

train_df = train_df.rename(columns={"polarization": "label"})
val_df   = val_df.rename(columns={"polarization": "label"})
test_df  = test_df.rename(columns={"polarization": "label"})

for df in [train_df, val_df, test_df]:
    df["label"] = df["label"].astype(int)


In [5]:
train_df.columns

Index(['id', 'text', 'label'], dtype='object')

In [6]:
from datasets import DatasetDict, Dataset

dataset = DatasetDict({
    "train": Dataset.from_pandas(train_df),
    "validation": Dataset.from_pandas(val_df),
    "test": Dataset.from_pandas(test_df),
})

dataset = dataset.remove_columns("__index_level_0__")

In [7]:
def tokenize(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True)

In [8]:
dataset = dataset.map(tokenize, batched=True)

Map:   0%|          | 0/2577 [00:00<?, ? examples/s]

Map:   0%|          | 0/322 [00:00<?, ? examples/s]

Map:   0%|          | 0/323 [00:00<?, ? examples/s]

In [9]:
from transformers import DistilBertForSequenceClassification

model = DistilBertForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=2)

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [10]:
metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    # convert the logits to their predicted class
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

In [11]:
training_args = TrainingArguments(
    output_dir=os.path.join(get_project_root(), 'predictions', 'finetuning'),
    eval_strategy="epoch",
    push_to_hub=False,
)

In [12]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    compute_metrics=compute_metrics,
)
trainer.train()

D:\projects\mldl\semevalpolar\.venv\Lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 